<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #10b981; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Preprocesamiento para Dummies: Gramos vs Elefantes (Escalado) 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #10b981; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Ruta Didáctica: Para Dummies 🧸 | Guía Didáctica
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #059669; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 01 • Data Mining
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #10b981; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Mining/01%20-%20Preprocesamiento%20de%20los%20Datos/Para%20Dummies/02_Escalado_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

En el Cuaderno 00 (Imputación) rellenamos los huecos del dataset. Pero incluso con todos los huecos rellenos, hay otro problema escondido: cada columna numérica está medida en una unidad distinta. `ingreso_mensual` se mide en pesos colombianos y se mueve en los millones; `tasa_interes` se mide en puntos porcentuales y se mueve entre 1 y 5. Si le entregas esas columnas "crudas" a un modelo que compara números entre sí, vas a tener un problema serio — y en este cuaderno vas a comprobar exactamente por qué, con números reales, y cómo se arregla.

---
## 1. Gramos vs Elefantes: cuando la unidad hace trampa 🐘

Imagina que quieres comparar el tamaño de un elefante y el tamaño de una hormiga, pero a quien hace la medición se le ocurre medir al elefante en **kilómetros** y a la hormiga en **milímetros**. El elefante mide "0.003 km" y la hormiga mide "5 mm". Si solo miras los números sin pensar en las unidades, ¡la hormiga (5) parece más grande que el elefante (0.003)! Por supuesto que eso es absurdo: el problema no es el tamaño real de ninguno de los dos, es que usaste una regla distinta para cada uno.

Exactamente lo mismo le pasa a nuestro dataset de crédito. `ingreso_mensual` está en pesos y se mueve en los **millones** — piensa en el elefante. `tasa_interes` está en puntos porcentuales y se mueve apenas entre **1 y 5** — piensa en la hormiga. Si un modelo compara estas dos columnas "a lo crudo", va a asumir que `ingreso_mensual` es un millón de veces más importante que `tasa_interes`, no porque realmente lo sea para el negocio, sino solo porque a alguien se le ocurrió medirlo en pesos en vez de, digamos, millones de pesos. **Escalar** las variables es ponerlas a todas con la misma regla, para que el modelo compare peras con peras.

---
## 2. Vamos a los Datos Reales 🔍

Igual que en el cuaderno anterior, trabajamos con `dataset_crudo_preprocesamiento.csv`. Esta vez nos importa el **rango** de cada columna numérica: qué tan chico es el valor más pequeño y qué tan grande es el más grande.

In [ ]:
import os, urllib.request, urllib.parse, gzip, base64

_DATA_EMBEDDED = {
    "dataset_crudo_preprocesamiento.csv": "..."
}

def load_dataset(filename="dataset_crudo_preprocesamiento.csv", module_name="01 - Preprocesamiento de los Datos"):
    """
    Carga o descarga de forma segura el dataset para ejecución local o en Google Colab.
    Si no se encuentra localmente ni en GitHub, lo restaura automáticamente de forma exacta.
    """
    candidates = [
        os.path.join("data", filename),
        os.path.join(module_name, "data", filename),
        os.path.join("..", "data", filename),
        os.path.join("..", module_name, "data", filename),
        os.path.join("..", "..", "data", filename),
        os.path.join("Data Mining", module_name, "data", filename),
        os.path.join("..", "Data Mining", module_name, "data", filename),
        filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path

    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    folder_path = f"Data Mining/{module_name}"
    encoded_folder = urllib.parse.quote(folder_path)
    encoded_file = urllib.parse.quote(filename)

    urls = [
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{encoded_folder}/data/{encoded_file}",
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/master/{encoded_folder}/data/{encoded_file}"
    ]

    print(f"[INFO] Intentando descargar '{filename}' desde el repositorio oficial...")
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status == 200:
                    with open(target_path, "wb") as out_f:
                        out_f.write(response.read())
                    print(f"[OK] Dataset '{filename}' descargado exitosamente.")
                    return target_path
        except Exception:
            continue

    if filename in _DATA_EMBEDDED:
        print(f"[INFO] Restaurando '{filename}' de forma autonoma...")
        data_bytes = gzip.decompress(base64.b64decode(_DATA_EMBEDDED[filename]))
        with open(target_path, "wb") as out_f:
            out_f.write(data_bytes)
        print(f"[OK] Dataset '{filename}' preparado exitosamente en '{target_path}'.")
        return target_path

    raise FileNotFoundError(f"No se pudo encontrar ni generar el dataset '{filename}'.")


import pandas as pd
import numpy as np
import os

ruta = load_dataset("dataset_crudo_preprocesamiento.csv")

df = pd.read_csv(ruta)

columnas_numericas = ["edad", "ingreso_mensual", "score_credito", "tasa_interes", "monto_prestamo"]
print("Rango (minimo - maximo) de cada columna numerica:\n")
for columna in columnas_numericas:
    print(f"  {columna:<18} va de {df[columna].min():>15,.2f}  a  {df[columna].max():>15,.2f}")

### 🤔 ¿Qué acaba de pasar?
`tasa_interes` apenas se mueve entre 1.2 y 4.5, y `score_credito` entre 394 y 850. Pero `ingreso_mensual` llega hasta 95,000,000 y `monto_prestamo` hasta 546,672,629. Ahí están, con números reales, nuestro elefante (`ingreso_mensual` y `monto_prestamo`) y nuestra hormiga (`tasa_interes`). (`edad` va de -3 a 150 — valores imposibles que se estudian a fondo en el Cuaderno 01; aquí no los tocamos todavía.)

---
## 3. La Prueba con Números: una Distancia que Solo le Cree a los Millones 🔬

Cuando un algoritmo como KNN o K-Means quiere decidir si dos clientes "se parecen", calcula algo parecido a una distancia: resta sus valores columna por columna, eleva al cuadrado, suma todo y saca raíz cuadrada. Vamos a hacerlo a mano con dos clientes reales, usando solo `ingreso_mensual` y `tasa_interes`, **sin escalar nada todavía**.

In [ ]:
pares = df.dropna(subset=["ingreso_mensual", "tasa_interes"])[["ingreso_mensual", "tasa_interes"]]
cliente_a = pares.iloc[0]
cliente_b = pares.iloc[1]

diferencia_ingreso = cliente_a["ingreso_mensual"] - cliente_b["ingreso_mensual"]
diferencia_tasa = cliente_a["tasa_interes"] - cliente_b["tasa_interes"]

aporte_ingreso = diferencia_ingreso ** 2
aporte_tasa = diferencia_tasa ** 2
distancia = np.sqrt(aporte_ingreso + aporte_tasa)

print(f"Cliente A: ingreso=${cliente_a['ingreso_mensual']:,.0f}  |  tasa={cliente_a['tasa_interes']:.2f}%")
print(f"Cliente B: ingreso=${cliente_b['ingreso_mensual']:,.0f}  |  tasa={cliente_b['tasa_interes']:.2f}%")
print(f"\nDistancia total entre A y B (sin escalar): {distancia:,.2f}")
print(f"% de esa distancia que le debe a la diferencia de INGRESO: {aporte_ingreso / (aporte_ingreso + aporte_tasa) * 100:.8f}%")
print(f"% de esa distancia que le debe a la diferencia de TASA:    {aporte_tasa / (aporte_ingreso + aporte_tasa) * 100:.8f}%")

### 🤔 ¿Qué acaba de pasar?
Prácticamente el 100% de la distancia se explica por la diferencia de ingreso; la diferencia de 1.43 puntos en la tasa de interés —que en la vida real sí importa para saber qué tan caro es un préstamo— queda totalmente invisible para el cálculo. No porque sea poco importante, sino solo por su unidad: es la hormiga contra el elefante otra vez.

---
## 4. `StandardScaler`: resta el promedio y divide entre qué tan disperso está 📊

`StandardScaler` hace algo muy simple: a cada valor le resta el promedio de su columna, y el resultado lo divide entre la desviación estándar (una medida de qué tan "esparcidos" están los datos). El resultado queda **centrado en 0**, con la mayoría de los valores moviéndose aproximadamente entre -3 y 3 — sin importar si la columna original estaba en pesos, en puntos porcentuales o en cualquier otra unidad. Repitamos la distancia de la Sección 3, pero esta vez con las dos columnas ya escaladas.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador_std = StandardScaler()
pares_escalados = escalador_std.fit_transform(pares)

print("Promedio y desviacion estandar ANTES de escalar:")
print(pares.agg(["mean", "std"]))
print("\nPromedio y desviacion estandar DESPUES de StandardScaler (deben quedar en ~0 y ~1):")
print(pd.DataFrame(pares_escalados, columns=pares.columns).agg(["mean", "std"]))

diferencia_ingreso_z = pares_escalados[0, 0] - pares_escalados[1, 0]
diferencia_tasa_z = pares_escalados[0, 1] - pares_escalados[1, 1]
distancia_z = np.sqrt(diferencia_ingreso_z ** 2 + diferencia_tasa_z ** 2)

print(f"\nDistancia entre Cliente A y B CON StandardScaler: {distancia_z:.4f}")
print(f"% que le debe ahora a la tasa de interes: {diferencia_tasa_z**2 / (diferencia_ingreso_z**2 + diferencia_tasa_z**2) * 100:.2f}%")

### 🤔 ¿Qué acaba de pasar?
La tasa de interés pasó de explicar una fracción invisible de la distancia a explicar cerca del **34.5%** de ella — una participación mucho más justa, dado que ambas columnas aportan información real de negocio. `StandardScaler` no cambió qué tan "distintos" son los clientes; solo dejó de dejarse engañar por la unidad en la que venía cada columna.

---
## 5. `MinMaxScaler`: aprieta todo entre 0 y 1, como un termómetro 🌡️

`MinMaxScaler` toma el valor más chico de la columna y lo convierte en 0, toma el más grande y lo convierte en 1, y reparte todo lo demás proporcionalmente en medio — como un termómetro que siempre marca entre 0 y 100, sin importar si mide el clima de Tunja o el interior de un congelador. Suena perfecto... hasta que un solo valor rarísimo se convierte en el "100" de ese termómetro y aplasta a todo el resto cerca del "0". Vamos a comprobarlo con `monto_prestamo`, que tiene un préstamo de ¡$546,672,629!

In [ ]:
from sklearn.preprocessing import MinMaxScaler

monto = df[["monto_prestamo"]].fillna(df["monto_prestamo"].median())
escalador_mm = MinMaxScaler()
monto_mm = escalador_mm.fit_transform(monto)

print(f"monto_prestamo ANTES  -> minimo: ${monto['monto_prestamo'].min():,.0f}  |  maximo: ${monto['monto_prestamo'].max():,.0f}")
print(f"monto_prestamo DESPUES de MinMaxScaler -> minimo: {monto_mm.min():.4f}  |  maximo: {monto_mm.max():.4f}")

porcentaje_apretujado = (monto_mm < 0.05).mean() * 100
print(f"\n% de clientes cuyo monto_prestamo quedo por debajo de 0.05 (practicamente pegado al 0): {porcentaje_apretujado:.1f}%")

### 🤔 ¿Qué acaba de pasar?
El rango final es exactamente [0, 1], tal como promete la fórmula. Pero más del **81%** de los préstamos quedaron apretujados en el primer 5% de ese rango, porque el "100" del termómetro lo fijó un único cliente con un préstamo extraordinario. Para casi cualquier algoritmo, esto vuelve casi imposible distinguir un préstamo pequeño de uno mediano.

---
## 6. `RobustScaler`: usa la mediana para no dejarse engañar por rarezas 🛡️

En vez de usar el promedio y la desviación estándar (que un solo valor rarísimo puede desviar muchísimo), `RobustScaler` usa la **mediana** (el valor de en medio) y el rango donde vive el 50% central de los datos. Ya viste en el Cuaderno 01 ("El Gigante de Tres Metros") que unos pocos outliers pueden desviar por completo un promedio; `RobustScaler` es la versión "de escalado" de esa misma lección: la mediana casi no se mueve aunque el valor máximo sea disparatado.

In [ ]:
from sklearn.preprocessing import RobustScaler

escalador_rob = RobustScaler()
monto_robusto = escalador_rob.fit_transform(monto)

escalador_std2 = StandardScaler()
monto_estandar = escalador_std2.fit_transform(monto)

en_rango_estandar = ((monto_estandar >= -1) & (monto_estandar <= 1)).mean() * 100
en_rango_robusto = ((monto_robusto >= -1) & (monto_robusto <= 1)).mean() * 100

print(f"Con StandardScaler, {en_rango_estandar:.1f}% de los clientes caen apretujados entre -1 y 1")
print(f"Con RobustScaler,   {en_rango_robusto:.1f}% de los clientes caen en ese mismo rango")
print(f"\nValor del cliente con el prestamo mas alto (el outlier) bajo StandardScaler: {monto_estandar.max():.2f}")
print(f"Valor del cliente con el prestamo mas alto (el outlier) bajo RobustScaler:   {monto_robusto.max():.2f}")

### 🤔 ¿Qué acaba de pasar?
Bajo `StandardScaler`, el 95.4% de los clientes quedan apretujados entre -1 y 1 — no porque todos tengan préstamos parecidos, sino porque el outlier infló tanto la "dispersión típica" que todo lo demás se ve comprimido cerca de 0. Bajo `RobustScaler` solo el 82.1% cae en ese mismo rango: el 50% central de los clientes queda repartido en un intervalo mucho más fiel a su propia variación real, aunque el outlier en sí termine todavía más lejos (31.97 contra 13.89). `RobustScaler` no "esconde" al outlier — simplemente no deja que ese único cliente aplaste a todos los demás.

---
## 7. Manos a la Obra: Tu Turno 📝

**Contexto:** `tasa_interes` no tiene ningún valor nulo y es una columna bastante simétrica (sin outliers extremos) — un buen caso para comparar escaladores sin la complicación de valores rarísimos de por medio.

**Tu misión:**
1. Crea un `MinMaxScaler` y un `StandardScaler`.
2. Ajusta y transforma la columna `tasa_interes` con cada uno (pista: usa doble corchete, `df[["tasa_interes"]]`).
3. Imprime el rango (mínimo, máximo) que deja `MinMaxScaler` y el promedio/desviación estándar que deja `StandardScaler`.
4. En un comentario dentro de tu celda, responde: si vas a alimentar esta columna a una red neuronal cuya primera capa espera valores entre 0 y 1, ¿cuál de los dos escaladores es la elección natural?

In [ ]:
### TU CODIGO AQUI ###
# 1. Crea los dos escaladores
escalador_mm_ej = None
escalador_std_ej = None

# 2. Ajusta y transforma tasa_interes con cada uno
tasa_mm = None
tasa_std = None

# 3. Imprime el rango de MinMaxScaler y el promedio/desviacion de StandardScaler
print("MinMaxScaler   -> minimo:", None, "maximo:", None)
print("StandardScaler -> promedio:", None, "desviacion:", None)

# 4. Respuesta en comentario:
# ...

<details>
<summary>💡 Ver solución propuesta</summary>

```python
# 1. Crea los dos escaladores
escalador_mm_ej = MinMaxScaler()
escalador_std_ej = StandardScaler()

# 2. Ajusta y transforma tasa_interes con cada uno
tasa_mm = escalador_mm_ej.fit_transform(df[["tasa_interes"]])
tasa_std = escalador_std_ej.fit_transform(df[["tasa_interes"]])

# 3. Imprime el rango de MinMaxScaler y el promedio/desviacion de StandardScaler
print("MinMaxScaler   -> minimo:", tasa_mm.min(), "maximo:", tasa_mm.max())
print("StandardScaler -> promedio:", tasa_std.mean(), "desviacion:", tasa_std.std())

# 4. Respuesta:
# MinMaxScaler es la eleccion natural, porque garantiza un rango exacto [0, 1] --
# justo lo que espera esa primera capa. StandardScaler no promete ningun minimo
# ni maximo fijo (solo promedio 0 y desviacion 1), asi que no encaja tan bien
# con una entrada que debe quedar estrictamente entre 0 y 1.
```

Con `tasa_interes` (sin nulos y sin outliers extremos), `MinMaxScaler` deja el rango exacto `[0.0, 1.0]` y `StandardScaler` deja un promedio de `~0` con desviación estándar `~1`. Ninguno de los dos "rompe" la columna, porque no hay valores rarísimos que compliquen las cosas — la elección entre ambos depende de lo que espera el algoritmo, no de que uno sea "mejor" en abstracto.

</details>

---
## 8. Resumen Relámpago ⚡

| Idea Clave | Explicación en 5 segundos |
|---|---|
| **Escalar** | Poner todas las columnas en la misma "regla" para que ninguna domine solo por su unidad. |
| **`StandardScaler`** | Resta el promedio y divide entre la dispersión: la columna queda centrada en 0. |
| **`MinMaxScaler`** | Aprieta todo entre 0 y 1, como un termómetro — pero un solo outlier puede aplastar al resto. |
| **`RobustScaler`** | Usa la mediana y el rango central en vez del promedio: los valores rarísimos no lo confunden. |
| **Regla de oro** | El algoritmo no sabe de pesos ni de porcentajes: si no escalas, la columna con los números más grandes se roba toda la atención. |

**Siguiente paso:** en el Cuaderno 03, **"Traduciendo Palabras a Números (Encoding)"**, vas a aprender a convertir las columnas de texto (`escolaridad`, `canal_contacto`) en números que el modelo sí pueda entender.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Ruta Para Dummies: Preprocesamiento</i>
  </p>
</div>